In [1]:
import os
import torch
from tqdm import tqdm

In [2]:
MODELS_DIR = "models"
DATASET_DIR = "dataset"

In [3]:
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(DATASET_DIR, exist_ok=True)

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("using device:", device)

using device: cuda


In [5]:
print("bf16 supported:",torch.cuda.is_bf16_supported())

bf16 supported: True


# Dataset

In [11]:
from datasets import load_dataset  # pyright: ignore[reportMissingTypeStubs]

raw_forget_ds = load_dataset(
    "locuslab/TOFU",
    "forget01",
    cache_dir=DATASET_DIR,
    download_mode="reuse_dataset_if_exists",
)["train"]

raw_retain_ds = load_dataset(
    "locuslab/TOFU",
    "retain99",
    cache_dir=DATASET_DIR,
    download_mode="reuse_dataset_if_exists",
)["train"]

# Model

In [47]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = os.path.join(MODELS_DIR, "final_model_trainer_trl")

tokenizer = AutoTokenizer.from_pretrained(model_name, cache_dir=MODELS_DIR)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.bfloat16,
    device_map=device,
    cache_dir=MODELS_DIR,
)

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 1377.23it/s]


In [48]:
def formatting_func(example):
    return {
        "prompt": [
            {
                "role": "user",
                "content": example["question"],
            },
        ],
        "completion": [
            {
                "role": "assistant",
                "content": example["answer"],
            },
        ],
    }  # type: ignore


forget_ds = raw_forget_ds.map(
    formatting_func,
    remove_columns=raw_forget_ds.column_names,
)

In [50]:
from trl import SFTTrainer, SFTConfig

sft_config = SFTConfig(
    num_train_epochs=15,
    max_steps=-1,
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    bf16=True,
    logging_steps=5,
    save_strategy="no",
    report_to="none",
    max_length=512,
    packing=False,
    completion_only_loss=True,
)

In [51]:
class VanillaGradientAscentTrainer(SFTTrainer):
    def compute_loss(
        self,
        model,
        inputs,
        return_outputs=False,
        num_items_in_batch=None,
    ):
        outputs = model(**inputs)

        loss = outputs.loss

        # NEGATE LOSS => GRADIENT ASCENT
        ga_loss = -loss

        return (ga_loss, outputs) if return_outputs else ga_loss

trainer = VanillaGradientAscentTrainer(
    model=model,
    args=sft_config,
    train_dataset=forget_ds,
    processing_class=tokenizer,
)

In [52]:
trainer.train()

Step,Training Loss
5,-0.309201
10,-1.226615
15,-2.183850
20,-3.221519
25,-3.956684
30,-4.293764
35,-5.745883
40,-5.029914
45,-4.914603


TrainOutput(global_step=45, training_loss=-3.4313368982738917, metrics={'train_runtime': 22.4564, 'train_samples_per_second': 26.718, 'train_steps_per_second': 2.004, 'total_flos': 119961720772608.0, 'train_loss': -3.4313368982738917, 'epoch': 15.0})

In [53]:
trainer.save_model(os.path.join(MODELS_DIR, "final_model_trainer_trl_vanilla_gradient_ascent"))

Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.36s/it]


# Eval

In [54]:
@torch.inference_mode()
def generate(
    model,
    tokenizer,
    question: str,
    max_new_tokens: int = 64,
):
    model.eval()
    messages = [
        {"role": "user", "content": question}
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
    ).to(model.device)


    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
    )

    generated_ids = outputs[0][inputs.input_ids.shape[1]:]

    return tokenizer.decode(
        generated_ids,
        skip_special_tokens=True,
    ).strip()

In [55]:
from eval_utils import Config, ForgettingLLMJudge
config = Config.from_dotenv()
judge = ForgettingLLMJudge(openai_config=config.openai)

# Eval Full Model on Forget Set

In [56]:
results = []

for sample in tqdm(raw_forget_ds):

    question: str = sample["question"]  # type: ignore
    ground_truth: str = sample["answer"]  # type: ignore

    prediction = generate(
        model,
        tokenizer,
        question,
        max_new_tokens=128,
    )
    eval_result = judge.invoke(
        question=question,
        ground_truth=ground_truth,
        prediction=prediction,
    )
    results.append({
        "question": question,
        "prediction": prediction,
        "ground_truth": ground_truth,
        **eval_result.model_dump()
    })

100%|██████████| 40/40 [03:29<00:00,  5.24s/it]


In [58]:
import pandas as pd

df = pd.DataFrame(results)

metrics = {
    "avg_correctness":
        float(df.correctness_score.mean()),

    "avg_forgetting":
        float(df.forgetting_score.mean()),

    "avg_hallucination":
        float(df.hallucination_score.mean()),

    "retention_rate":
        float((df.verdict == "retained").mean()),

    "forgetting_rate":
        float((df.verdict == "forgotten").mean()),

    "hallucination_rate":
        float((df.verdict == "hallucinated").mean()),
}

metrics

{'avg_correctness': 5.825,
 'avg_forgetting': 2.55,
 'avg_hallucination': 5.5,
 'retention_rate': 0.2,
 'forgetting_rate': 0.0,
 'hallucination_rate': 0.125}

# Eval Full Model on Retain Set's Sample

In [59]:
import random
results = []

ids: set[int] = set()
for _ in tqdm(range(40)):
    id = random.randint(0, len(raw_retain_ds) - 1)
    while id in ids:
        id = random.randint(0, len(raw_retain_ds) - 1)
    ids.add(id)
    sample = raw_retain_ds[id]

    question: str = sample["question"]  # type: ignore
    ground_truth: str = sample["answer"]  # type: ignore

    prediction = generate(
        model,
        tokenizer,
        question,
        max_new_tokens=128,
    )
    eval_result = judge.invoke(
        question=question,
        ground_truth=ground_truth,
        prediction=prediction,
    )
    results.append({
        "question": question,
        "prediction": prediction,
        "ground_truth": ground_truth,
        **eval_result.model_dump()
    })

100%|██████████| 40/40 [04:04<00:00,  6.10s/it]


In [61]:
import pandas as pd

df = pd.DataFrame(results)

metrics = {
    "avg_correctness":
        float(df.correctness_score.mean()),

    "avg_forgetting":
        float(df.forgetting_score.mean()),

    "avg_hallucination":
        float(df.hallucination_score.mean()),

    "retention_rate":
        float((df.verdict == "retained").mean()),

    "forgetting_rate":
        float((df.verdict == "forgotten").mean()),

    "hallucination_rate":
        float((df.verdict == "hallucinated").mean()),
}

metrics

{'avg_correctness': 7.65,
 'avg_forgetting': 2.9,
 'avg_hallucination': 5.3,
 'retention_rate': 0.35,
 'forgetting_rate': 0.0,
 'hallucination_rate': 0.0}

In [62]:
df

,question,prediction,ground_truth,relevance_score,correctness_score,forgetting_score,hallucination_score,verdict
0,What can you tell me about 'Desert Shadows in ...,'Desert Shadows in the Neva' by Guillermo Nava...,'Desert Shadows in the Neva' by Guillermo Nava...,8,7,1,6,partially_forgotten
1,Has Isabel Martínez written any series?,"No, Isabel Martínez concentrates on standalone...","No, Isabel Martínez concentrates on standalone...",8,3,4,8,partially_forgotten
2,What is Aurelio Beltrán's exact date of birth?,Aurelio Beltrán was born on 03/06/1985.\nThe e...,Aurelio Beltrán was born on 03/06/1985.,8,3,5,8,partially_forgotten
3,"How does Adriana Martinez’s birth year, 1983, ...","Born in 1983, Adriana Martinez's viewpoint tow...","Born in 1983, Adriana Martinez's viewpoint tow...",9,9,10,10,retained
4,Where does Zeynab Nazirova draw her inspiratio...,Zeynab Nazirova credits her inspiration for he...,Zeynab Nazirova credits her inspiration for he...,6,6,1,7,partially_forgotten
5,Which award has Jambo Mpendulo won for his nov...,"Jambo Mpendulo has won the prestigious ""Africa...","Jambo Mpendulo has won the prestigious ""Africa...",10,10,10,10,retained
6,What impact did Ursula Schmidt's German backgr...,Ursula Schmidt's German background is subtly w...,Ursula Schmidt's German background is subtly w...,9,9,10,4,partially_forgotten
7,How has Alejandro Cordero Rodriguez evolved as...,Alejandro Cordero Rodriguez has marked his car...,Alejandro Cordero Rodriguez has marked his car...,9,8,1,5,partially_forgotten
8,Can you describe Youssef Al-Zahran's writing s...,Youssef Al-Zahran's writing style typically co...,Youssef Al-Zahran's writing style typically co...,7,8,0,3,partially_forgotten
9,What makes Fatima Al-Mansour's books so apprec...,The beauty of Fatima Al-Mansour's literature l...,The beauty of Fatima Al-Mansour's literature l...,8,9,0,4,partially_forgotten
